In [ ]:
import numpy as np
import pandas as pd

: 

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator

In [ ]:
from bokeh.plotting import figure, show, output_file
from bokeh.io import output_notebook
import bokeh.models as bm
output_notebook(hide_banner=True)

: 

In [ ]:
def sims_matrix(rows, cols, mean, stdev, seed=1):
    """Create a matrix of simulation adding noise (random numbers) about the mean and standard devation 'stdev'. Rows contain time and columns contain different simulations.
    """
    np.random.seed(seed)
    x = np.random.randn(rows, cols)
    x = mean + (x * stdev)
    return x

def growth(principal, rate, n, contribution):
    """Calculate new balance over 1-year time step with annual compounding interest and additional regular contributions. Agrees with https://www.moneygeek.com/compound-interest-calculator/ if n_p=n.

    Inputs:
        principal: float, starting account balance
        rate: float, average interest rate expressed as a decimal
        n: int or float, number of contributions. For monthly, use n=12.
        contribution: float, amount to be contributed n times.
    
    Output:
       New account balance after interest and contributions
    """
    t = 1
    compound_interest = principal * (1 + (rate / n)) ** (n * t)
    contributions = contribution * (((1 + (rate / n)) ** (n * t) - 1) / (rate / n))
    return np.round(compound_interest + contributions, 2)


def withdrawal_simulation(start_capital, return_mean, return_stdev, inflation_mean, inflation_stdev, monthly_withdrawal, n_years=30, n_simulations=100):
    """Simulate monthly withdrawals from a starting investment amount under different inflation and growth scenarios.
    
    Inputs:
        start_capital: float, starting investment amount
        return_mean: float, average rate of return on investment as a percent
        return_stdev: float, standard deviation for return on investment
        inflation_mean: float, average rate of inflation as a percent
        inflation_stdev: float, standard deviation of inflation
        monthly_withdrawal: float, amount to be withdrawn each month
        n_years: int, number of years to simulate drawdown
        n_simulations: int, number of simulations to generate
    
    Output:
        Dataframe of account balances over time (rows) for each simulation (columns).
    """
    # Convert mean percents to decimals
    if return_mean > 1:
        return_mean = return_mean / 100
    if inflation_mean > 1:
        inflation_mean = inflation_mean / 100

    # Convert annual values to monthly (using volitility square root of time for stdev)
    n_months = 12 * n_years
    # monthly_return_mean = return_mean / 12
    monthly_return_mean = ((return_mean + 1) ^ (1/12)) - 1
    monthly_return_stdev = return_stdev / np.sqrt(12)
    # monthly_inflation_mean = inflation_mean / 12
    monthly_inflation_mean = ((inflation_mean + 1) ^ (1/12)) - 1
    monthly_inflation_stdev = inflation_stdev / np.sqrt(12)

    # Simulate returns and inflation
    monthly_returns = sims_matrix(
        rows=n_months,
        cols=n_simulations,
        mean=monthly_return_mean,
        stdev=monthly_return_stdev)
    monthly_inflation = sims_matrix(
        rows=n_months,
        cols=n_simulations, 
        mean=monthly_inflation_mean,
        stdev=monthly_inflation_stdev)
    monthly_withdrawal = sims_matrix(
        rows=n_months,
        cols=n_simulations,
        mean=monthly_withdrawal,
        stdev=0.05)

    # Simulate withdrawals
    sims = np.full((n_months + 1, n_simulations), float(start_capital))
    for j in range(n_months):
        sims[j + 1, :] = (
            sims[j, :] *
            (1 + monthly_returns[j, :] - monthly_inflation[j, :]) -
            monthly_withdrawal[j, :]
        )

    # Set sims values below 0 to NaN
    sims[sims < 0] = np.nan

    # convert to millions
    sims = sims / 1000000

    return pd.DataFrame(sims)


def growth_plot(nav_df):
    # Create the figure
    p = figure(
        title='Projected retirement savings balance by year (age)',
        tools='pan, wheel_zoom, box_zoom, undo, reset, fullscreen',
        outline_line_color=None, sizing_mode='scale_height'
        )
    source = bm.ColumnDataSource(nav_df)

    # Create x-axis in years
    thisYear = pd.to_datetime('today').year
    xlabs = np.linspace(0, nav_df.shape[0]/12, num=nav_df.shape[0]) + thisYear


    # Plot the simulations
    for i, column in enumerate(nav_df.columns):
        lp = p.line(y=column, name=f'Simulation {i+1}')


def growth_plot_static(nav_df):
    # Create the figure and axes
    fig, ax1 = plt.subplots(1, 1)

    # Create x-axis in years
    thisYear = pd.to_datetime('today').year
    xlabs = np.linspace(0, nav_df.shape[0]/12, num=nav_df.shape[0]) + thisYear

    for column in nav_df.columns:
        ax1.plot(xlabs, nav_df[column], alpha=0.3)

    ax1.set_xlim(min(xlabs), max(xlabs))
    ax1.xaxis.set_minor_locator(AutoMinorLocator())
    xlabels = [item.get_text() for item in ax1.get_xticklabels()]
    xlabels = [l+f'\n({float(l)-1986})' for l in xlabels]
    ax1.set_xticklabels(xlabels)
    ax1.yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter('${x:,.0f}'))
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.title.set_text('Projected retirement savings balance by year (age)')
    ax1.grid(True, alpha=0.5)

    plt.tight_layout()

    return fig

def withdrawal_plot(nav_df, scenario_percent, retire_age):
    # For the histogram, we will fill NaNs with -1
    nav_df_zeros = nav_df.ffill().fillna(0).iloc[-1, :]

    # Define the figure and axes
    fig = plt.figure()

    # Create the top plot for time series on the first row that spans all columns
    ax1 = plt.subplot2grid((2, 2), (0, 0), colspan=2)

    # Create the bottom left plot for the percentage above zero
    ax2 = plt.subplot2grid((2, 2), (1, 0), colspan=2)

    # Create x-axis in years
    xlabs = np.linspace(0, nav_df.shape[0]/12, num=nav_df.shape[0]) + retire_age

    for column in nav_df.columns:
        ax1.plot(xlabs, nav_df[column], alpha=0.3)

    ax1.set_xlim(min(xlabs), max(xlabs))
    ax1.xaxis.set_minor_locator(AutoMinorLocator())
    ax1.yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter('${x:,.0f}'))
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.title.set_text(f"Projected value of capital over {int(nav_df[1:].shape[0]/12)} years")
    ax1.grid(True, alpha=0.5)

    # Calculate the percentage of columns that are above zero for each date and plot (bottom left plot)
    percent_above_zero = (nav_df > 0).sum(axis=1) / nav_df.shape[1] * 100
    ax2.plot(
        xlabs[percent_above_zero>scenario_percent],
        percent_above_zero[percent_above_zero>scenario_percent], color='darkgreen', linewidth=2)
    ax2.plot(
        xlabs[percent_above_zero<scenario_percent], percent_above_zero[percent_above_zero<scenario_percent], color='darkred', linewidth=2)
    ax2.plot(
        xlabs[(percent_above_zero>=scenario_percent-2.5) & (percent_above_zero<=scenario_percent+2.5)], 
        percent_above_zero[(percent_above_zero>=scenario_percent-2.5) & (percent_above_zero<=scenario_percent+2.5)], color='khaki', linewidth=2)
    ax2.set_xlim(min(xlabs), max(xlabs))
    ax2.xaxis.set_minor_locator(AutoMinorLocator())
    ax2.set_ylim(0, 105)  # Percentage goes from 0 to 100 with buffer
    ax2.yaxis.set_major_formatter('{x:.0f}%')
    ax2.title.set_text("Percent of scenarios still paying out")
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.set_xlabel("Age")
    ax2.grid(True, alpha=0.5)
    ax2.fill_between(x=xlabs, y1=percent_above_zero, color="red", 
        alpha=0.2, where=percent_above_zero<scenario_percent-2.5)
    ax2.fill_between(x=xlabs, y1=percent_above_zero, color="green", 
        alpha=0.2, where=percent_above_zero>scenario_percent+2.5)
    ax2.fill_between(
        x=xlabs[(percent_above_zero>=scenario_percent-2.5) & (percent_above_zero<=scenario_percent+2.5)], 
        y1=percent_above_zero[(percent_above_zero>=scenario_percent-2.5) & (percent_above_zero<=scenario_percent+2.5)], color='yellow', alpha=0.2)

    plt.tight_layout()

    return fig


In [ ]:
def growth_simulation(start_capital, return_mean, return_stdev, raise_mean, raise_stdev, monthly_contribution, n_years=30, n_simulations=100):
    """Simulate investment growth from a starting amount under different pay raises and growth scenarios and with monthly contributions. The output of this calculator agrees, to a few cents, with The Calculator Site,
    https://www.thecalculatorsite.com/finance/calculators/compoundinterestcalculator.php.
    
    Inputs:
        start_capital: float, starting investment amount
        return_mean: float, average rate of return on investment as a percent
        return_stdev: float, standard deviation for return on investment
        raise_mean: float, average annual pay raise as a percent.    
            Used to adjust contributions.
        raise_stdev: float, standard deviation of annual pay raises
        monthly_contribution: float, amount to be contributed each month
        n_years: int, number of years to simulate drawdown
        n_simulations: int, number of simulations to generate
    
    Output:
        Dataframe of account balances over time (rows) for each simulation (columns).
    """
    # Convert mean and standard deviations from percents to decimals
    return_mean = return_mean / 100
    return_stdev = return_stdev / 100
    raise_mean = raise_mean / 100
    raise_stdev = raise_stdev / 100

    # Convert annual values to monthly (using volitility square root of time for stdev)
    n_months = 12 * n_years
    monthly_return_mean = return_mean / 12
    # monthly_return_mean = ((return_mean + 1) ** (1/12)) - 1
    monthly_return_stdev = return_stdev / np.sqrt(12)

    # Simulate returns and raises
    monthly_returns = sims_matrix(
        rows=n_months+12,
        cols=n_simulations,
        mean=monthly_return_mean,
        stdev=monthly_return_stdev)
    raises = sims_matrix(
        rows=n_years,
        cols=n_simulations, 
        mean=raise_mean+1,
        stdev=raise_stdev)

    # Contributions adjusted for annual inflation raises
    contributions = np.full((n_years+1, n_simulations),
                            float(monthly_contribution))
    for j in range(n_years):
        contributions[j+1, :] = contributions[j, :] * raises[j, :]
    contributions = np.repeat(contributions, 12, axis=0)
    contributions = np.concatenate((np.zeros((1, n_simulations)), contributions), axis=0)

    # Simulate growth
    sims = np.full((n_months+12, n_simulations), float(start_capital))
    for j in range(n_months+11):
        sims[j+1, :] = \
            growth(principal=sims[j, :],
                   rate=monthly_returns[j, :], n=1,
                   contribution=contributions[j+1, :])

    return pd.DataFrame(sims[:-11,:])


In [14]:
from bokeh.models import NumeralTickFormatter, Range1d, LinearAxis, CustomJSTickFormatter

In [ ]:
def growth_plot(nav_df, start_year=None):
    # Create the figure
    p = figure(
        title='Projected retirement savings balance by year (age)',
        tools='pan, wheel_zoom, box_zoom, undo, reset, fullscreen',
        outline_line_color=None, sizing_mode='scale_height'
        )

    # Create x-axis in years
    if start_year == None:
        thisYear = pd.to_datetime('today').year
    else:
        thisYear = int(start_year)
    xlabs = np.linspace(0, nav_df.shape[0]/12, num=nav_df.shape[0]) + thisYear
    nav_df.columns = [f'sim{i+1}' for i in nav_df.columns]
    nav_df['xlabs'] = xlabs
    source = bm.ColumnDataSource(nav_df)

    # Plot the simulations
    for i, column in enumerate(nav_df.columns):
        if column != 'xlabs':
            lp = p.line(x='xlabs', y=column, name=f'Simulation {i+1}', source=source)
    
    # Format axes
    # xlabels = [item.get_text() for item in p.get_xticklabels()]
    # xlabels = [l+f'\n({float(l)-1986})' for l in xlabels]
    # p.set_xticklabels(xlabels)
    p.yaxis.formatter=NumeralTickFormatter(format='$0,0')

    show(p)



In [ ]:
growth_df = growth_simulation(
    start_capital=300000,
    return_mean=5,
    return_stdev=5,
    raise_mean=1.,
    raise_stdev=1.,
    monthly_contribution=2250,
    n_years=30,
    n_simulations=250)

start_capital=200000
return_mean=5
return_stdev=5
raise_mean=1
raise_stdev=1
monthly_contribution=2000
n_years=25
n_simulations=250
start_year=None

nav_df = growth_df

p = figure(
        title='Projected retirement savings balance by year (age)',
        tools='pan, wheel_zoom, box_zoom, undo, reset, fullscreen',
        outline_line_color=None)#, sizing_mode='scale_height')

# Name simulations for legend
nav_df.columns = [f'sim{i+1}' for i in nav_df.columns]

# Create x-axis in years
if start_year == None:
    thisYear = pd.to_datetime('today').year
else:
    thisYear = int(start_year)
xlabs = np.linspace(0, nav_df.shape[0]/12, num=nav_df.shape[0]) + thisYear
nav_df['xlabs'] = xlabs

# Plot the simulations
source = bm.ColumnDataSource(nav_df)
# for i, column in enumerate(nav_df.columns):
#     if column != 'xlabs':
#         lp = p.line(x='xlabs', y=column, name=f'Simulation {i+1}',
#                     alpha=0.1, source=source)

# Plot the average
nav_df['average'] = nav_df.drop('xlabs', axis=1).mean(1)
nav_df['age'] = np.floor(nav_df['xlabs'] - 1986)
source = bm.ColumnDataSource(nav_df)
al = p.line(x='xlabs', y='average', color='blue', width=2, name='Average', source=source)
p.varea(x='xlabs', y1=0, y2='average', color='lightblue', alpha=0.5, source=source)

# Tools
crosshair = bm.CrosshairTool(dimensions='height',
                            line_color='grey', line_alpha=0.5)
hover = bm.HoverTool(mode='vline', renderers=[al])
hover.tooltips = """
        <h2>${x}{0} | Age @{age}{0}</h2>
        @{average}{$0,0.00}
    """

p.add_tools(hover, crosshair)
p.toolbar.autohide = True


# Format axes
p.xaxis.formatter = CustomJSTickFormatter(code="""
        return tick + " (" + (tick-1986) + ")"
    """)
p.yaxis.formatter=NumeralTickFormatter(format='$0,0')
p.xgrid.grid_line_color = None

show(p)



: 

In [22]:
import scipy.stats as st

In [23]:
np.apply_along_axis(nav_df, )
# st.t.interval(0.95, len(a)-1, loc=np.mean(a), scale=st.sem(a))

Object `nav.df.apply` not found.


In [153]:
def ci(data, confidence=0.95):
        
    mn = data.mean(axis=1)
    se = data.sem(axis=1)
    n = data.shape[1]
    h = se * st.t.ppf((1 + confidence) / 2., n-1, loc=mn, scale=se)
    print(se)

    lower = mn-h
    upper = mn+h

    return h

    return data[data.apply(lambda row: (row>=lower) & (row<=upper))].mean(axis=1)

    return mn-h, mn, mn+h

In [121]:
ci(data=nav_df.iloc[:,:-3], confidence=0.90)

0      2.000000e+05
1      2.016741e+05
2      2.033461e+05
3      2.050254e+05
4      2.067100e+05
           ...     
296    1.038918e+06
297    1.043322e+06
298    1.047800e+06
299    1.052302e+06
300    1.056857e+06
Length: 301, dtype: float64

In [110]:
((1+0.95)/2)-0.95

0.025000000000000022

In [116]:
confidence = 0.95
data = nav_df.iloc[:,:-3]
mn = data.mean(axis=1)
se = data.sem(axis=1)
n = data.shape[1]
h = se * st.t.ppf((1 + confidence) / 2., df=n-1)

lower = mn-h
upper = mn+h

# data[data.apply(lambda row: (row>=lower) & (row<=upper))].isna().sum(1)

In [117]:
[st.t.ppf((1 + confidence) / 2., df=n-1, loc=m, scale=s) for m,s in zip(mn, se)]

/opt/conda/lib/python3.11/site-packages/scipy/stats/_distn_infrastructure.py:2285: RuntimeWarning: invalid value encountered in multiply
  lower_bound = _a * scale + loc
/opt/conda/lib/python3.11/site-packages/scipy/stats/_distn_infrastructure.py:2286: RuntimeWarning: invalid value encountered in multiply
  upper_bound = _b * scale + loc


[nan,
 201678.89994640046,
 203354.9436799821,
 205040.1031053509,
 206729.97588215917,
 208419.29563971263,
 210115.9388393791,
 211823.97000332133,
 213527.8980115034,
 215238.74828906474,
 216963.6061618745,
 218698.4207166462,
 220417.952720281,
 222156.20507736751,
 223896.03414249973,
 225644.7827858614,
 227400.20393100003,
 229159.460106285,
 230929.44954182795,
 232699.04881105066,
 234477.8250244345,
 236265.1601620781,
 238061.52428084507,
 239866.71256748482,
 241676.21222023148,
 243492.55775443668,
 245306.70268966406,
 247118.9554226102,
 248952.23688163536,
 250793.45728363443,
 252630.9971909341,
 254480.0182967312,
 256332.5653024207,
 258194.23857795136,
 260059.83046439505,
 261928.39811742,
 263799.2880914408,
 265678.3538092015,
 267560.3214180462,
 269447.80830455694,
 271354.1361039791,
 273256.394332195,
 275163.55398946145,
 277091.06937024766,
 279011.4743905358,
 280943.91934165347,
 282887.7374038466,
 284840.2935241021,
 286788.851074234,
 288757.836317384

In [118]:
[st.t.ppf(((1 + confidence) / 2.)-0.95, df=n-1, loc=m, scale=s) for m,s in zip(mn, se)]

/opt/conda/lib/python3.11/site-packages/scipy/stats/_distn_infrastructure.py:2285: RuntimeWarning: invalid value encountered in multiply
  lower_bound = _a * scale + loc
/opt/conda/lib/python3.11/site-packages/scipy/stats/_distn_infrastructure.py:2286: RuntimeWarning: invalid value encountered in multiply
  upper_bound = _b * scale + loc


[nan,
 201656.8490135996,
 203324.5820800179,
 205003.72401464914,
 206688.64459784087,
 208371.4614002873,
 210063.10836062088,
 211765.16639667863,
 213466.09566849662,
 215174.06371093524,
 216894.72687812545,
 218622.4736833538,
 220335.351919719,
 222073.07652263253,
 223806.9858575003,
 225551.65697413858,
 227303.26622899994,
 229059.85861371504,
 230826.47245817207,
 232594.65606894932,
 234369.51041556548,
 236154.12047792197,
 237946.1001191549,
 239747.16551251517,
 241552.6105797686,
 243361.46224556334,
 245173.33731033592,
 246980.93489738987,
 248811.15935836468,
 250643.67527636563,
 252478.11400906593,
 254321.35026326877,
 256169.18325757928,
 258027.03766204868,
 259886.90409560502,
 261750.93700258006,
 263614.85982855933,
 265489.71387079847,
 267364.62954195385,
 269248.090415443,
 271146.61229602084,
 273040.9577478049,
 274940.2154505386,
 276862.9028697524,
 278777.6332094642,
 280703.1358583465,
 282641.0875561534,
 284587.59887589794,
 286528.86444576597,
 28

In [70]:
data[data.apply(lambda row: (row>=lower) & (row<=upper))].mean(axis=1)

0      2.000000e+05
1      2.016777e+05
2      2.032862e+05
3      2.049444e+05
4      2.065759e+05
           ...     
296    1.039811e+06
297    1.043476e+06
298    1.048311e+06
299    1.053080e+06
300    1.057375e+06
Length: 301, dtype: float64

In [291]:
start_capital=200000
return_mean=4
return_stdev=0
raise_mean=1
raise_stdev=0
monthly_contribution=1000
n_years=25
n_simulations=1

In [292]:
    # Convert mean percents to decimals
    return_mean = return_mean / 100
    return_stdev = return_stdev / 100
    raise_mean = raise_mean / 100
    raise_stdev = raise_stdev / 100


In [293]:
return_mean

0.04

In [294]:
    # Convert annual values to monthly (using volitility square root of time for stdev)
    n_months = 12 * n_years
    monthly_return_mean = return_mean / 12
    monthly_return_stdev = return_stdev / np.sqrt(12)


In [295]:
monthly_return_mean

0.0033333333333333335

In [334]:
    # Simulate returns and raises
    monthly_returns = sims_matrix(
        rows=n_months+12,
        cols=n_simulations,
        mean=monthly_return_mean,
        stdev=monthly_return_stdev)
    # yearly_returns = sims_matrix(
    #     rows=n_years,
    #     cols=n_simulations,
    #     mean=return_mean,
    #     stdev=return_stdev)
    raises = sims_matrix(
        rows=n_years,
        cols=n_simulations, 
        mean=raise_mean+1,
        stdev=raise_stdev)
    # raises = np.concatenate((np.ones((1, n_simulations)), raises), axis=0)
    # raises = np.repeat(raises, 12, axis=0)


In [335]:
print(raises.shape)
print(raises)

(25, 1)
[[1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]
 [1.01]]


In [336]:
monthly_returns.shape

(312, 1)

In [337]:
raises.shape

(25, 1)

In [338]:
    # Contributions adjusted for annual inflation raises
    contributions = np.full((n_years+1, n_simulations),
                            float(monthly_contribution))
    for j in range(n_years):
        contributions[j+1, :] = (
            contributions[j, :] * raises[j, :]
        )
    contributions = np.repeat(contributions, 12, axis=0)
    contributions = np.concatenate((np.zeros((1, n_simulations)), contributions), axis=0)
    print(contributions.shape)
    print(contributions)
    

(313, 1)
[[   0.        ]
 [1000.        ]
 [1000.        ]
 [1000.        ]
 [1000.        ]
 [1000.        ]
 [1000.        ]
 [1000.        ]
 [1000.        ]
 [1000.        ]
 [1000.        ]
 [1000.        ]
 [1000.        ]
 [1010.        ]
 [1010.        ]
 [1010.        ]
 [1010.        ]
 [1010.        ]
 [1010.        ]
 [1010.        ]
 [1010.        ]
 [1010.        ]
 [1010.        ]
 [1010.        ]
 [1010.        ]
 [1020.1       ]
 [1020.1       ]
 [1020.1       ]
 [1020.1       ]
 [1020.1       ]
 [1020.1       ]
 [1020.1       ]
 [1020.1       ]
 [1020.1       ]
 [1020.1       ]
 [1020.1       ]
 [1020.1       ]
 [1030.301     ]
 [1030.301     ]
 [1030.301     ]
 [1030.301     ]
 [1030.301     ]
 [1030.301     ]
 [1030.301     ]
 [1030.301     ]
 [1030.301     ]
 [1030.301     ]
 [1030.301     ]
 [1030.301     ]
 [1040.60401   ]
 [1040.60401   ]
 [1040.60401   ]
 [1040.60401   ]
 [1040.60401   ]
 [1040.60401   ]
 [1040.60401   ]
 [1040.60401   ]
 [1040.60401   ]
 [104

In [339]:
contributions.shape

(313, 1)

In [349]:
    # Simulate growth
    sims = np.full((n_months+12, n_simulations), float(start_capital))
    for j in range(n_months+11):
        sims[j+1, :] = \
            growth(principal=sims[j, :],
                   rate=monthly_returns[j, :], n=1,
                   contribution=contributions[j+1, :])


In [350]:
print(sims.shape)
print(sims[:-11,0])

(312, 1)
[ 200000.    201666.67  203338.89  205016.69  206700.08  208389.08
  210083.71  211783.99  213489.94  215201.57  216918.91  218641.97
  220370.78  222115.35  223865.73  225621.95  227384.02  229151.97
  230925.81  232705.56  234491.25  236282.89  238080.5   239884.1
  241693.71  243519.46  245351.29  247189.23  249033.29  250883.5
  252739.88  254602.45  256471.22  258346.22  260227.47  262114.99
  264008.81  265919.14  267835.84  269758.93  271688.43  273624.36
  275566.74  277515.6   279470.95  281432.82  283401.23  285376.2
  287357.76  289356.22  291361.34  293373.15  295391.66  297416.9
  299448.89  301487.66  303533.22  305585.6   307644.82  309710.91
  311783.88  313874.17  315971.43  318075.68  320186.94  322305.24
  324430.6   326563.05  328702.6   330849.29  333003.13  335164.15
  337332.37  339518.33  341711.58  343912.14  346120.03  348335.28
  350557.92  352787.97  355025.45  357270.39  359522.81  361782.74
  364050.2   366335.84  368629.09  370929.99  373238.56  

This matches with https://www.thecalculatorsite.com/finance/calculators/compoundinterestcalculator.php